# Mastercard AI Defence Lab - Final Submission

This notebook is the single Kaggle GPU entry point for the synthetic adaptive red-team/blue-team experiment. It uses Qwen2.5-7B agents, conditional CTGAN generation, a continual fraud detector, a fixed unseen seven-family benchmark, and auditable trend gates. Measured values are never replaced or fabricated.

## 1. Configure Kaggle GPU Environment and Paths

The paths and run controls below can be overridden with environment variables. The experiment runs one five-round qualification gate before one independent 50-round submission run.

In [2]:
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY_URL = os.getenv(
    "MASTERCARD_REPOSITORY_URL",
    "https://github.com/keshav-0210/mastercard_hackathon.git",
)
PROJECT_DIR = Path(os.getenv("MASTERCARD_PROJECT_DIR", "/kaggle/working/mastercard_hackathon"))
KAGGLE_INPUT_DIR = Path(os.getenv("KAGGLE_INPUT_DIR", "/kaggle/input"))
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
SUBMISSION_DIR = Path(os.getenv("SUBMISSION_DIR", "/kaggle/working/mastercard_submission"))
RUN_ROUNDS = 50
GATE_ROUNDS = 5
SEED = 20260822
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))

print("Project:", PROJECT_DIR)
print("Artifacts:", ARTIFACTS_DIR)
print("Submission bundle:", SUBMISSION_DIR)
print("Gate rounds:", GATE_ROUNDS)
print("Submission rounds:", RUN_ROUNDS)
print("Run timestamp:", RUN_STAMP)

Cloning into '/kaggle/working/mastercard_hackathon'...


Project: /kaggle/working/mastercard_hackathon
Artifacts: /kaggle/working/mastercard_hackathon/artifacts
Submission bundle: /kaggle/working/mastercard_submission
Gate rounds: 5
Submission rounds: 50
Run timestamp: 20260829T140936Z


In [3]:
required_source_markers = {
    PROJECT_DIR / "src" / "mastercard_defence" / "detector.py": "calibration_data",
    PROJECT_DIR / "src" / "mastercard_defence" / "loop.py": "fixed_unseen_seven_family_benchmark",
    PROJECT_DIR / "src" / "mastercard_defence" / "synthetic.py": "family_coverage_diversity_ratio",
    PROJECT_DIR / "src" / "mastercard_defence" / "submission.py": "assess_submission_trends",
    PROJECT_DIR / "src" / "ui" / "generate_dashboard.py": "blue_team_benchmark_roc_auc",
}
stale_sources = []
for path, marker in required_source_markers.items():
    if not path.exists() or marker not in path.read_text(encoding="utf-8"):
        stale_sources.append(f"{path.relative_to(PROJECT_DIR)} missing {marker!r}")
if stale_sources:
    raise RuntimeError(
        "The cloned repository is older than this final notebook. Sync the local source changes "
        "to the repository, restart the Kaggle session, and rerun. Details: " + "; ".join(stale_sources)
    )
print("Final source revision check: passed")

Final source revision check: passed


## 2. Install and Import Dependencies

Install only the runtime packages used by the final experiment. The CUDA 12.4 llama.cpp wheel is required for Qwen GPU offload; package versions are captured for the artifact manifest.

In [4]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        "ctgan==0.11.0",
        "numpy>=1.26,<3",
        "pandas>=2.1,<3",
        "pydantic>=2.7,<3",
        "scikit-learn>=1.4,<2",
        "PyYAML>=6,<7",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        "--prefer-binary",
        "--index-url",
        "https://abetlen.github.io/llama-cpp-python/whl/cu124",
        "--extra-index-url",
        "https://pypi.org/simple",
        "llama-cpp-python>=0.2.90",
    ],
    check=True,
)

import importlib.metadata as metadata
import json
import random
import shutil
import time
import zipfile

import numpy as np
import pandas as pd
import torch

PACKAGE_NAMES = [
    "ctgan",
    "llama-cpp-python",
    "numpy",
    "pandas",
    "pydantic",
    "scikit-learn",
    "PyYAML",
    "torch",
]
PACKAGE_VERSIONS = {name: metadata.version(name) for name in PACKAGE_NAMES}
print(json.dumps(PACKAGE_VERSIONS, indent=2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 867.8 kB/s eta 0:00:00
{
  "ctgan": "0.11.0",
  "llama-cpp-python": "0.3.35",
  "numpy": "2.0.2",
  "pandas": "2.3.3",
  "pydantic": "2.12.3",
  "scikit-learn": "1.6.1",
  "PyYAML": "6.0.3",
  "torch": "2.10.0+cu128"
}


## 3. Validate GPU and Reproducibility Settings

Require llama.cpp GPU offload for Qwen, seed Python, NumPy, and Torch, and locate the two attached GGUF shards. CTGAN uses Torch CUDA only when the installed wheel contains kernels for the attached GPU architecture; otherwise CTGAN trains on CPU while Qwen remains fully GPU-offloaded.

In [5]:
from llama_cpp import llama_supports_gpu_offload

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is unavailable. Enable a GPU accelerator before continuing.")
if not llama_supports_gpu_offload():
    raise RuntimeError("llama.cpp lacks CUDA offload support; refusing a CPU or heuristic Qwen fallback.")

device_capability = torch.cuda.get_device_capability(0)
torch_gpu_arch = f"sm_{device_capability[0]}{device_capability[1]}"
torch_compiled_arches = set(torch.cuda.get_arch_list())
ctgan_cuda = torch_gpu_arch in torch_compiled_arches
if ctgan_cuda:
    torch.cuda.manual_seed_all(SEED)

gguf_files = sorted(KAGGLE_INPUT_DIR.rglob("*.gguf"))
if len(gguf_files) != 2:
    raise FileNotFoundError(f"Expected exactly two Qwen GGUF shards, found: {gguf_files}")
model_path = next(
    (path for path in gguf_files if "00001-of-00002" in path.name),
    None,
)
if model_path is None:
    raise FileNotFoundError("The first Qwen GGUF shard was not found")

os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)
os.environ["GPU_LAYERS"] = "-1"

print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("Torch GPU architecture:", torch_gpu_arch)
print("Torch compiled architectures:", sorted(torch_compiled_arches))
print("CTGAN device:", "cuda:0" if ctgan_cuda else "cpu")
print("Qwen shards:", [path.name for path in gguf_files])
print("llama.cpp Qwen GPU offload: enabled")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 16269 MiB):
  Device 0: Tesla P100-PCIE-16GB, compute capability 6.0, VMM: yes, VRAM: 16269 MiB


CUDA: 12.8
GPU count: 1
GPUs: ['Tesla P100-PCIE-16GB']
Torch GPU architecture: sm_60
Torch compiled architectures: ['sm_100', 'sm_120', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
CTGAN device: cpu
Qwen shards: ['qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf', 'qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf']
llama.cpp Qwen GPU offload: enabled


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


## 4. Load and Validate Fraud Data

This challenge implementation is synthetic-only. It does not load real payment or cardholder records. The cell below generates the approved legitimate reference schema and validates labels, missing values, ranges, and feature exclusions.

## 5. Prepare Features and Fraud Labels

The production pipeline adds synthetic fraud classes and performs categorical encoding inside `FraudDetector`. It creates disjoint training, calibration, current-family holdout, and fixed seven-family benchmark sets. The target and provenance fields are explicitly excluded from detector features.

In [6]:
from mastercard_defence.detector import FEATURES
from mastercard_defence.synthetic import make_reference_transactions

reference_sample = make_reference_transactions(2_000, SEED)
required_columns = set(FEATURES) | {"is_fraud"}
missing_columns = sorted(required_columns - set(reference_sample.columns))
if missing_columns:
    raise ValueError(f"Synthetic reference data is missing columns: {missing_columns}")
if reference_sample[list(required_columns)].isna().any().any():
    raise ValueError("Synthetic reference data contains missing required values")
if "is_fraud" in FEATURES or "attack_family" in FEATURES:
    raise ValueError("Target or attack-family provenance leaked into detector features")
if not set(reference_sample["is_fraud"].unique()).issubset({0, 1}):
    raise ValueError("Fraud labels must be binary")

print("Rows:", len(reference_sample))
print("Detector features:", FEATURES)
print("Fraud-label mean:", float(reference_sample["is_fraud"].mean()))
print("Missing required values: 0")
print("Target leakage check: passed")

Rows: 2000
Detector features: ['amount', 'hour', 'device_change', 'beneficiary_change', 'velocity_24h', 'channel']
Fraud-label mean: 0.0
Missing required values: 0
Target leakage check: passed


## 6. Configure Qwen-Assisted CTGAN Experiment

    "Use one shared lazy-loaded Qwen instance for both runs to avoid loading two 7B models into GPU memory. CTGAN is fit within each robustness suite. The detector threshold is calibrated only on a separate legitimate holdout and held to one flat 5.0% operating-target ceiling every round; benchmark labels never influence calibration."

In [7]:
from copy import deepcopy

from mastercard_defence.agents import ATTACK_FAMILIES, QwenAgents
from mastercard_defence.llm import SharedLocalLLM
from mastercard_defence.loop import ClosedLoop, load_config
from mastercard_defence.submission import assess_submission_trends, write_submission_artifacts
from mastercard_defence.synthetic import build_metrics_dump

config = load_config(str(PROJECT_DIR / "config" / "default.yaml"))
config["run_mode"] = "KAGGLE_GPU"
config["seed"] = SEED
config["paths"]["memory_db"] = str(ARTIFACTS_DIR / f"adaptive_v2_memory_{RUN_STAMP}.sqlite")
config["generator_backend"] = "ctgan"
config["generator_cuda"] = ctgan_cuda
config["generator_epochs"] = 50
config["generator_training_attack_size"] = 40
config["generator_training_reference_size"] = 400
config["detector_mode"] = "continual"
config["detector_retrain_every"] = 2
config["detector_target_fpr"] = 0.05
config["max_historical_pool"] = 800
config["max_replay_examples"] = 800
config["weakness_weight_multiplier"] = 4.0
config["benchmark_attacks_per_family"] = 40
config["threshold_calibration_rows"] = 5_000
config["model"]["path"] = str(model_path)
config["model"]["context_size"] = 4_096
config["model"]["max_output_tokens"] = 256
config["model"]["gpu_layers"] = -1
config["pipeline"]["rounds"] = RUN_ROUNDS
config["pipeline"]["synthetic_transactions"] = 400
config["pipeline"]["max_generated_attacks"] = 80
config["pipeline"]["fraud_rate"] = 0.02

if len(ATTACK_FAMILIES) != 7:
    raise ValueError(f"Expected seven approved fraud families, found {ATTACK_FAMILIES}")

shared_llm = SharedLocalLLM(config)
agents = QwenAgents(config, llm=shared_llm)
serializable_config = deepcopy(config)
print(json.dumps(serializable_config, indent=2))
print("Approved families:", list(ATTACK_FAMILIES))
print("Agent backend:", type(agents).__name__)

{
  "run_mode": "KAGGLE_GPU",
  "seed": 20260822,
  "model": {
    "path": "/kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "repo": "Qwen/Qwen2.5-7B-Instruct-GGUF",
    "filename": "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "shard_count": 2,
    "context_size": 4096,
    "max_output_tokens": 256,
    "gpu_layers": -1
  },
  "paths": {
    "knowledge_base": "data/knowledge_base",
    "artifacts": "artifacts",
    "memory_db": "/kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_memory_20260829T140936Z.sqlite"
  },
  "pipeline": {
    "rounds": 50,
    "synthetic_transactions": 400,
    "max_generated_attacks": 80,
    "fraud_rate": 0.02,
    "rag_top_k": 4,
    "detector_train_fraction": 0.6
  },
  "generator_backend": "ctgan",
  "generator_cuda": false,
  "generator_epochs": 50,
  "generator_training_attack_size": 40,
  "generator_training_reference_size": 400,
  "detector_mode": "continual",
  "det

## 7. Train CTGAN for 50 Rounds

First run the five-round fixed-benchmark qualification. The 50-round Qwen+CTGAN experiment is a fresh run and is code-blocked unless F1, Recall, Precision, and ROC-AUC increase while false-positive rate decreases across the qualification windows. CTGAN is trained once per suite, not once per round.

In [8]:
gate_config = deepcopy(config)
gate_memory_path = ARTIFACTS_DIR / f"adaptive_v2_gate_memory_{RUN_STAMP}.sqlite"
gate_memory_path.unlink(missing_ok=True)
gate_config["paths"]["memory_db"] = str(gate_memory_path)
gate_config["pipeline"]["rounds"] = GATE_ROUNDS

gate_loop = ClosedLoop(gate_config, agents=agents)
gate_started = time.perf_counter()
try:
    gate_suite = gate_loop.run_robustness_suite(seeds=1, rounds=GATE_ROUNDS)
finally:
    gate_loop.close()

gate_results = [result for run in gate_suite["by_seed"] for result in run["results"]]
gate_metrics = build_metrics_dump(gate_results)
gate_assessment = assess_submission_trends(
    gate_metrics,
    window_size=2,
    include_red_team=False,
)
print("Five-round duration seconds:", round(time.perf_counter() - gate_started, 2))
print(json.dumps(gate_assessment, indent=2))
if not gate_assessment["passed"]:
    raise RuntimeError("Five-round fixed-benchmark trend gate failed; 50-round run is blocked")

[robustness] preparing CTGAN training corpus
[robustness] fitting CTGAN: rows=680 epochs=50
[robustness] CTGAN ready in 15.18s; entering round loop
[robustness] starting suite: seeds=1, rounds=5
[robustness] starting seed 20260822 with family plan: ['account_takeover', 'cross_channel_anomaly', 'social_engineering', 'beneficiary_manipulation', 'low_and_slow']
[ctgan] family='account_takeover' attempt=3/32 matched=24/40
[ctgan] family='account_takeover' attempt=4/32 matched=33/40
[ctgan] family='account_takeover' attempt=5/32 matched=35/40
[ctgan] family='account_takeover' attempt=6/32 matched=39/40
[ctgan] family='trusted_device' attempt=3/32 matched=26/40
[ctgan] family='trusted_device' attempt=4/32 matched=29/40
[ctgan] family='trusted_device' attempt=5/32 matched=35/40
[ctgan] family='beneficiary_manipulation' attempt=3/32 matched=29/40
[ctgan] family='beneficiary_manipulation' attempt=4/32 matched=34/40
[ctgan] family='beneficiary_manipulation' attempt=5/32 matched=35/40
[ctgan] fam

In [9]:
import contextlib
import io


class Tee(io.TextIOBase):
    def __init__(self, *streams):
        self.streams = streams

    def write(self, text):
        for stream in self.streams:
            stream.write(text)
            stream.flush()
        return len(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()


full_memory_path = Path(config["paths"]["memory_db"])
full_memory_path.unlink(missing_ok=True)
log_path = ARTIFACTS_DIR / f"adaptive_v2_run_{RUN_STAMP}.log"
full_loop = ClosedLoop(config, agents=agents)
full_started = time.perf_counter()
with log_path.open("w", encoding="utf-8") as log_file:
    with contextlib.redirect_stdout(Tee(sys.stdout, log_file)):
        with contextlib.redirect_stderr(Tee(sys.stderr, log_file)):
            print(
                "RUN_CONFIGURATION",
                json.dumps(
                    {
                        "seeds": 1,
                        "rounds": RUN_ROUNDS,
                        "agent_backend": "QwenAgents",
                        "generator_backend": "conditional_ctgan",
                        "detector_mode": config["detector_mode"],
                        "fraud_rate": config["pipeline"]["fraud_rate"],
                    }
                ),
            )
            try:
                suite = full_loop.run_robustness_suite(seeds=1, rounds=RUN_ROUNDS)
            finally:
                full_loop.close()

print("Fifty-round duration seconds:", round(time.perf_counter() - full_started, 2))
print("Run log:", log_path)
assert suite["seed_count"] == 1
assert suite["rounds"] == 50

RUN_CONFIGURATION {"seeds": 1, "rounds": 50, "agent_backend": "QwenAgents", "generator_backend": "conditional_ctgan", "detector_mode": "continual", "fraud_rate": 0.02}
[robustness] preparing CTGAN training corpus
[robustness] fitting CTGAN: rows=680 epochs=50
[robustness] CTGAN ready in 11.82s; entering round loop
[robustness] starting suite: seeds=1, rounds=50
[robustness] starting seed 20260822 with family plan: ['account_takeover', 'cross_channel_anomaly', 'social_engineering', 'beneficiary_manipulation', 'low_and_slow', 'trusted_device', 'merchant_abuse', 'account_takeover', 'cross_channel_anomaly', 'social_engineering', 'beneficiary_manipulation', 'low_and_slow', 'trusted_device', 'merchant_abuse', 'account_takeover', 'cross_channel_anomaly', 'social_engineering', 'beneficiary_manipulation', 'low_and_slow', 'trusted_device', 'merchant_abuse', 'account_takeover', 'cross_channel_anomaly', 'social_engineering', 'beneficiary_manipulation', 'low_and_slow', 'trusted_device', 'merchant_a

## 8. Run Red-Team and Blue-Team Evaluations

Every round already evaluates a disjoint generated attack holdout and the unchanged fixed seven-family benchmark. Current-family measurements feed Agent3; only fixed-benchmark measurements are used for longitudinal blue-team claims.

## 9. Calculate AUC-ROC and Core Metrics

`FraudDetector.evaluate` calculates ROC-AUC from `predict_proba` scores, never thresholded class predictions. The export retains fixed-benchmark Precision, Recall, F1, ROC-AUC, and false-positive rate.

## 10. Calculate Family Coverage Diversity

Family Coverage Diversity is the cumulative proportion of the seven approved fraud families reached so far. The final export uses `family_coverage_diversity_ratio` and excludes attack-channel diversity.

In [10]:
flattened_results = [result for run in suite["by_seed"] for result in run["results"]]
metrics = build_metrics_dump(flattened_results)

if len(metrics) != RUN_ROUNDS:
    raise ValueError(f"Expected {RUN_ROUNDS} metric rows, found {len(metrics)}")
for row in metrics:
    if row["blue_team_benchmark_protocol"] != "fixed_unseen_seven_family_benchmark":
        raise ValueError(f"Round {row['round']} used an invalid benchmark protocol")
    auc = row.get("blue_team_benchmark_roc_auc")
    if auc is None or not 0.0 <= float(auc) <= 1.0:
        raise ValueError(f"Round {row['round']} has invalid benchmark ROC-AUC: {auc}")
    if "attack_diversity_channel_entropy" in row:
        raise ValueError("Attack-channel diversity must not be exported")
    if "family_coverage_cumulative_ratio" in row:
        raise ValueError("Legacy family coverage field must not be exported")
    if row.get("family_coverage_diversity_ratio") is None:
        raise ValueError(f"Round {row['round']} lacks Family Coverage Diversity")

metric_preview_columns = [
    "round",
    "attack_family",
    "blue_team_benchmark_precision",
    "blue_team_benchmark_recall",
    "blue_team_benchmark_f1",
    "blue_team_benchmark_roc_auc",
    "blue_team_benchmark_false_positive_rate",
    "attack_novelty_score",
    "attack_fidelity_behavioural_plausibility",
    "family_coverage_diversity_ratio",
]
display(pd.DataFrame(metrics)[metric_preview_columns].tail(10))

,round,attack_family,blue_team_benchmark_precision,blue_team_benchmark_recall,blue_team_benchmark_f1,blue_team_benchmark_roc_auc,blue_team_benchmark_false_positive_rate,attack_novelty_score,attack_fidelity_behavioural_plausibility,family_coverage_diversity_ratio
40,41,cross_channel_anomaly,0.239474,0.325000,0.275758,0.863699,0.021064,0.8500,0.6456,1.0
41,42,cross_channel_anomaly,0.229765,0.314286,0.265460,0.861047,0.021501,0.8500,0.6519,1.0
42,43,cross_channel_anomaly,0.235294,0.285714,0.258065,0.862728,0.018950,0.8500,0.6214,1.0
43,44,cross_channel_anomaly,0.220708,0.289286,0.250386,0.860103,0.020845,0.8421,0.5962,1.0
44,45,low_and_slow,0.271795,0.378571,0.316418,0.869265,0.020700,0.8696,0.6540,1.0
45,46,cross_channel_anomaly,0.242775,0.300000,0.268371,0.866411,0.019096,0.8421,0.5933,1.0
46,47,beneficiary_manipulation,0.223587,0.325000,0.264920,0.859446,0.023032,0.9091,0.6036,1.0
47,48,low_and_slow,0.238916,0.346429,0.282799,0.859148,0.022522,0.8636,0.6509,1.0
48,49,social_engineering,0.236769,0.303571,0.266041,0.865763,0.019971,0.9091,0.6269,1.0
49,50,social_engineering,0.234211,0.317857,0.269697,0.861941,0.021210,0.9091,0.6316,1.0


## 11. Validate Expected Metric Trends

Compare the first and last ten rounds and calculate a full-run linear slope. Blue-team F1, Recall, Precision, and ROC-AUC must rise; false-positive rate must fall. Red-team fidelity must rise, all seven families must be covered without regression, and the final novelty window must remain above the configured floor. These checks report and preserve measured values unchanged.

In [11]:
final_assessment = assess_submission_trends(
    metrics,
    window_size=10,
    include_red_team=True,
    novelty_floor=0.7,
)
print(json.dumps(final_assessment, indent=2))
FINAL_TRENDS_PASSED = bool(final_assessment["passed"])
print("FINAL_TRENDS_PASSED", FINAL_TRENDS_PASSED)

{
  "passed": false,
  "rounds": 50,
  "window_size": 10,
  "metrics": {
    "blue_team_benchmark_f1": {
      "expectation": "increase",
      "first_window_mean": 0.265048,
      "last_window_mean": 0.271791,
      "absolute_change": 0.006743,
      "linear_slope": 0.00018,
      "passed": true
    },
    "blue_team_benchmark_recall": {
      "expectation": "increase",
      "first_window_mean": 0.317143,
      "last_window_mean": 0.318571,
      "absolute_change": 0.001429,
      "linear_slope": -7.2e-05,
      "passed": false
    },
    "blue_team_benchmark_precision": {
      "expectation": "increase",
      "first_window_mean": 0.228137,
      "last_window_mean": 0.237329,
      "absolute_change": 0.009192,
      "linear_slope": 0.000307,
      "passed": true
    },
    "blue_team_benchmark_roc_auc": {
      "expectation": "increase",
      "first_window_mean": 0.856951,
      "last_window_mean": 0.862955,
      "absolute_change": 0.006004,
      "linear_slope": 0.000195,
      "

## 12. Dump Metrics and Experiment Artifacts

Always persist raw results, the flat metric table, the gate evidence, and the final trend evidence. Also export CSV, the exact configuration, package versions, and the complete run log before deciding whether the dashboard is eligible for generation.

In [12]:
artifact_paths = write_submission_artifacts(
    suite,
    config,
    ARTIFACTS_DIR,
    RUN_STAMP,
    gate_assessment,
    final_assessment,
)
metrics_csv_path = ARTIFACTS_DIR / f"adaptive_v2_metrics_dump_{RUN_STAMP}.csv"
config_path = ARTIFACTS_DIR / f"adaptive_v2_config_{RUN_STAMP}.json"
versions_path = ARTIFACTS_DIR / f"adaptive_v2_package_versions_{RUN_STAMP}.json"

pd.DataFrame(metrics).to_csv(metrics_csv_path, index=False)
config_path.write_text(json.dumps(serializable_config, indent=2), encoding="utf-8")
versions_path.write_text(json.dumps(PACKAGE_VERSIONS, indent=2), encoding="utf-8")

print("Artifact paths:")
for path in [*artifact_paths.values(), metrics_csv_path, config_path, versions_path, log_path]:
    print(" -", path)

if not FINAL_TRENDS_PASSED:
    raise RuntimeError(
        "The raw 50-round artifacts were saved, but expected trends failed; dashboard generation is blocked"
    )

Artifact paths:
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_results_20260829T140936Z.json
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_metrics_dump_20260829T140936Z.json
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_summary_20260829T140936Z.json
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_metrics_dump_20260829T140936Z.csv
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_config_20260829T140936Z.json
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_package_versions_20260829T140936Z.json
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_run_20260829T140936Z.log


RuntimeError: The raw 50-round artifacts were saved, but expected trends failed; dashboard generation is blocked

## 13. Update and Copy the Submission Dashboard

Preserve the pre-run dashboard, rebuild the canonical page from the new 50-round JSON, and create a timestamped copy. The generator adds fixed-benchmark ROC-AUC, omits channel diversity, and labels Family Coverage Diversity consistently.

In [12]:
dashboard_path = ARTIFACTS_DIR / "submission_dashboard.html"
previous_dashboard_path = ARTIFACTS_DIR / f"submission_dashboard_before_{RUN_STAMP}.html"
if dashboard_path.exists():
    shutil.copy2(dashboard_path, previous_dashboard_path)

subprocess.run(
    [sys.executable, str(PROJECT_DIR / "src" / "ui" / "generate_dashboard.py")],
    cwd=PROJECT_DIR,
    check=True,
)
if not dashboard_path.exists():
    raise FileNotFoundError("Dashboard generator did not create submission_dashboard.html")

timestamped_dashboard_path = ARTIFACTS_DIR / f"submission_dashboard_{RUN_STAMP}.html"
local_download_dashboard_path = SUBMISSION_DIR / timestamped_dashboard_path.name
shutil.copy2(dashboard_path, timestamped_dashboard_path)
shutil.copy2(timestamped_dashboard_path, local_download_dashboard_path)
print("Canonical dashboard:", dashboard_path)
print("Timestamped dashboard:", timestamped_dashboard_path)
print("Download copy:", local_download_dashboard_path)

Dashboard written to /kaggle/working/mastercard_hackathon/artifacts/submission_dashboard.html
Metrics rows embedded: 50 (source: adaptive_v2_metrics_dump_20260829T095107Z.json)
Canonical dashboard: /kaggle/working/mastercard_hackathon/artifacts/submission_dashboard.html
Timestamped dashboard: /kaggle/working/mastercard_hackathon/artifacts/submission_dashboard_20260829T095107Z.html
Download copy: /kaggle/working/mastercard_submission/submission_dashboard_20260829T095107Z.html


## 14. Generate the README File

Create run-specific documentation inside the submission bundle with setup, data policy, metric definitions, outputs, reproducibility controls, and troubleshooting guidance.

In [13]:
submission_readme_path = SUBMISSION_DIR / "README.md"
submission_readme = f"""# Mastercard AI Defence Lab - Submission {RUN_STAMP}

## Purpose
This package contains evidence from a synthetic-only, offline adaptive fraud-defence experiment using Qwen2.5-7B agents, conditional CTGAN, and a continual HistGradientBoosting detector.

## Kaggle GPU setup
1. Upload `FINAL_SUBMISSION.ipynb` to Kaggle.
2. Enable a GPU accelerator.
3. Attach the private dataset containing both matching Qwen2.5-7B Q4_K_M GGUF shards.
4. Run all cells in order. No API key or runtime secret is required; the repository URL and paths can be overridden with environment variables.

## Execution
The notebook installs pinned core dependencies, validates CUDA and llama.cpp offload, runs a five-round fixed-benchmark gate, then runs exactly 50 fresh rounds only if the gate passes. Seed: {SEED}. Fraud rate: {config['pipeline']['fraud_rate']:.2%}.
Qwen must use llama.cpp GPU offload. CTGAN uses Torch CUDA only when the installed Torch wheel supports the attached GPU architecture; the selected CTGAN device is recorded in the exported configuration.
The detector threshold uses a separate legitimate-only calibration set and one flat 5.0% operating-target ceiling held constant across all 50 rounds. Benchmark labels never influence the threshold.

## Metrics
- Precision, Recall, F1, ROC-AUC, and false-positive rate use the same fixed unseen seven-family benchmark every round.
- ROC-AUC is calculated from probability scores.
- Family Coverage Diversity is the cumulative proportion of the seven approved fraud families explored.
- Attack Novelty measures structured distance from prior Attack Memory context.
- Attack Fidelity measures behavioural plausibility against the synthetic reference distribution.
- Solid dashboard lines are raw per-round values; dashed lines are trailing three-round rolling averages. Stored values are never smoothed or replaced.

## Outputs
The bundle includes raw results JSON, flat metrics JSON and CSV, summary/trend evidence, configuration, package versions, run log, checksums, and canonical/timestamped HTML dashboards.

## Reproducibility and safety
All transaction data is generated synthetically. No PII, cardholder data, production payment data, or live payment systems are used. The model shard hashes are not redistributed. The run records its seed, package versions, configuration, timestamp, and file hashes.

## Troubleshooting
- `CUDA unavailable`: enable a Kaggle GPU accelerator and restart the notebook.
- `Expected exactly two Qwen GGUF shards`: attach the complete two-shard model dataset.
- `llama.cpp lacks CUDA offload`: rerun the CUDA-wheel installation cell.
- `Five-round trend gate failed`: inspect the printed raw fixed-benchmark evidence; do not launch the 50-round run.
- `Final trend assessment failed`: use the saved JSON/CSV diagnostics. The notebook deliberately blocks dashboard generation rather than changing measured values.
"""
submission_readme_path.write_text(submission_readme, encoding="utf-8")
print("README:", submission_readme_path)

README: /kaggle/working/mastercard_submission/README.md


## 15. Safely Clean the Hackathon Folder

Cleanup is constrained to generated caches, temporary gate databases, and older adaptive-v2 outputs. It first prints a dry-run report and deletes only when `CONFIRM_SUBMISSION_CLEANUP=1`; source, data, rules, models, the notebook, README, current artifacts, architecture image, and both dashboards are always protected.

In [17]:
protected_paths = {
    path.resolve()
    for path in [
        *artifact_paths.values(),
        metrics_csv_path,
        config_path,
        versions_path,
        log_path,
        full_memory_path,
        dashboard_path,
        timestamped_dashboard_path,
        previous_dashboard_path,
        ARTIFACTS_DIR / "architecture_diagram.png",
        PROJECT_DIR / "FINAL_SUBMISSION.ipynb",
        PROJECT_DIR / "README.md",
    ]
    if path.exists()
}
cleanup_candidates = set(PROJECT_DIR.rglob("__pycache__"))
cleanup_candidates.update(PROJECT_DIR.rglob(".pytest_cache"))
cleanup_candidates.add(gate_memory_path)
for pattern in (
    "adaptive_v2_results_*.json",
    "adaptive_v2_metrics_dump_*.json",
    "adaptive_v2_metrics_dump_*.csv",
    "adaptive_v2_summary_*.json",
    "adaptive_v2_run_*.log",
    "adaptive_v2_memory_*.sqlite",
):
    cleanup_candidates.update(ARTIFACTS_DIR.glob(pattern))

cleanup_candidates = sorted(
    (path for path in cleanup_candidates if path.exists() and path.resolve() not in protected_paths),
    key=lambda path: (len(path.parts), str(path)),
    reverse=True,
)
print("CLEANUP_DRY_RUN")
for path in cleanup_candidates:
    print(" -", path)

CONFIRM_CLEANUP = os.getenv("CONFIRM_SUBMISSION_CLEANUP", "0") == "1"
if CONFIRM_CLEANUP:
    for path in cleanup_candidates:
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink(missing_ok=True)
    print("Cleanup completed:", len(cleanup_candidates), "generated paths removed")
else:
    print("Cleanup not executed. Set CONFIRM_SUBMISSION_CLEANUP=1 and rerun this cell after reviewing the list.")

CLEANUP_DRY_RUN
 - /kaggle/working/mastercard_hackathon/src/mastercard_defence/__pycache__
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_summary_20260828T202159Z.json
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_summary_20260828T143736Z.json
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_metrics_dump_20260828T202159Z.json
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_metrics_dump_20260828T143736Z.json
 - /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_gate_memory_20260829T095107Z.sqlite
Cleanup completed: 6 generated paths removed


## 16. Verify the End-to-End Submission Package

Validate JSON and CSV readability, required metric names, dashboard bindings, 50-round completeness, and source documentation. Then hash every deliverable and create one downloadable zip.

In [20]:
import hashlib

from IPython.display import FileLink, display

metrics_json = json.loads(artifact_paths["metrics"].read_text(encoding="utf-8"))
metrics_csv = pd.read_csv(metrics_csv_path)
if len(metrics_json) != 50 or len(metrics_csv) != 50:
    raise ValueError("Submission metrics must contain exactly 50 rounds")
if not all("blue_team_benchmark_roc_auc" in row for row in metrics_json):
    raise ValueError("Fixed-benchmark ROC-AUC is missing")
if any("attack_diversity_channel_entropy" in row for row in metrics_json):
    raise ValueError("Removed channel-diversity metric is still present")
if not all("family_coverage_diversity_ratio" in row for row in metrics_json):
    raise ValueError("Family Coverage Diversity is missing")

dashboard_html = dashboard_path.read_text(encoding="utf-8")
if "chart-blue_team_benchmark_roc_auc" not in dashboard_html:
    raise ValueError("Dashboard ROC-AUC chart is missing")
if "chart-attack_diversity_channel_entropy" in dashboard_html:
    raise ValueError("Dashboard still contains channel diversity")
if "Fraud-Family Coverage Diversity" not in dashboard_html:
    raise ValueError("Dashboard Family Coverage Diversity label is missing")

required_source_files = [PROJECT_DIR / "FINAL_SUBMISSION.ipynb", PROJECT_DIR / "README.md"]
for path in required_source_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required source file is absent from the cloned repository: {path}. Sync the final source before rerunning."
        )
architecture_path = ARTIFACTS_DIR / "architecture_diagram.png"
if not architecture_path.exists():
    raise FileNotFoundError("architecture_diagram.png is required beside the dashboard")

bundle_sources = [
    *artifact_paths.values(),
    metrics_csv_path,
    config_path,
    versions_path,
    log_path,
    dashboard_path,
    timestamped_dashboard_path,
    architecture_path,
    PROJECT_DIR / "FINAL_SUBMISSION.ipynb",
    (PROJECT_DIR / "README.md", "REPOSITORY_README.md"),
    submission_readme_path,
]
bundle_paths = []
for source_entry in bundle_sources:
    if isinstance(source_entry, tuple):
        source, destination_name = source_entry
    else:
        source = source_entry
        destination_name = source.name
    destination = SUBMISSION_DIR / destination_name
    if source.resolve() != destination.resolve():
        shutil.copy2(source, destination)
    bundle_paths.append(destination)

if len({path.name for path in bundle_paths}) != len(bundle_paths):
    raise ValueError("Submission bundle contains duplicate destination names")

manifest = {
    "created_at_utc": RUN_STAMP,
    "repository": REPOSITORY_URL,
    "seed": SEED,
    "rounds": RUN_ROUNDS,
    "five_round_gate_passed": gate_assessment["passed"],
    "final_trends_passed": final_assessment["passed"],
    "files": {
        path.name: {
            "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
        for path in bundle_paths
    },
}
manifest_path = SUBMISSION_DIR / f"manifest_{RUN_STAMP}.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
bundle_paths.append(manifest_path)

zip_path = Path("/kaggle/working") / f"mastercard_submission_{RUN_STAMP}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_paths:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(zip_path) as archive:
    archive_names = archive.namelist()
if len(archive_names) != len(set(archive_names)):
    raise ValueError("Submission zip contains duplicate entries")

print("END_TO_END_SUBMISSION_OK")
print("Files:", len(bundle_paths))
print("Zip:", zip_path)
display(FileLink(str(local_download_dashboard_path)))
display(FileLink(str(zip_path)))

END_TO_END_SUBMISSION_OK
Files: 14
Zip: /kaggle/working/mastercard_submission_20260829T095107Z.zip


/kaggle/working/mastercard_submission/submission_dashboard_20260829T095107Z.html

/kaggle/working/mastercard_submission_20260829T095107Z.zip

In [21]:
# Temporary transfer metadata; removed after the local dashboard copy is verified.
import base64
import gzip

dashboard_bytes = timestamped_dashboard_path.read_bytes()
compressed_dashboard = gzip.compress(dashboard_bytes, compresslevel=9, mtime=0)
dashboard_payload = base64.b64encode(compressed_dashboard).decode("ascii")
print("DASHBOARD_BYTES", len(dashboard_bytes))
print("COMPRESSED_BYTES", len(compressed_dashboard))
print("BASE64_LENGTH", len(dashboard_payload))
print("COMPRESSED_SHA256", hashlib.sha256(compressed_dashboard).hexdigest())

DASHBOARD_BYTES 92302
COMPRESSED_BYTES 16536
BASE64_LENGTH 22048
COMPRESSED_SHA256 9f7a3abb8f356a54dbe087d610ee2e1dbef2905fb9a38a0029d9cd25f2b4b154


In [22]:
print(dashboard_payload)

...
